# Capa Silver - Dimensión de Distribuidores

In [ ]:
from spark_session import *
from lakehouse_utils import *
from pyspark.sql import functions as f
from pyspark.sql.window import Window

## Spark Session

In [2]:
spark = create_spark_session(
    "retail-dlh-gold",
    configs=[
        configure_iceberg_delta_extensions,
        configure_delta_catalog,
        configure_polaris_restcatalog,
        configure_retail_polaris,
        configure_s3
    ]
)

## Extracción

In [3]:
target_table = "polaris.gold.DIM_DEALERSHIPS"
df_dealerships = spark.table("silver.dealerships")
df_dim_dealerships = spark.table(target_table)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Transformaciones

In [4]:
# Obtener valor maximo de la llave subrogada
max_sk = df_dim_dealerships.select(f.coalesce(f.max(f.col("DEALERSHIP_SK")), f.lit(0))).first()[0]

# Obtener registros nuevos y calcula su llave subrogada
df_dealerships_ins = df_dealerships.alias("d") \
    .join(df_dim_dealerships.alias("dd"), (f.col("d.DEALERSHIP_ID") == f.col("dd.DEALERSHIP_ID")), "left_anti") \
    .withColumn("DEALERSHIP_SK", max_sk + f.row_number().over(Window.orderBy(f.col("DEALERSHIP_ID")))) \
    .select(
        f.col("DEALERSHIP_SK"),
        f.col("d.*")
    )

# Obtener registros a actualizar
df_dealerships_upd = df_dealerships.alias("d") \
    .join(df_dim_dealerships.alias("dd"), (f.col("d.DEALERSHIP_ID") == f.col("dd.DEALERSHIP_ID")), "inner") \
    .select(
        f.col("dd.DEALERSHIP_SK"),
        f.col("d.*")
    )

# Seleccion de columnas finales
df_source = df_dealerships_ins.unionByName(df_dealerships_upd)

## Carga

In [5]:
# Obtenemos variables para merge
join_columns = ["DEALERSHIP_SK"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns)

## Cierre de SparkSession

In [6]:
spark.stop()